# Paridad de la lectura de la fracción: $F$ o $1-F$

**Objetivo.** Mostrar en funcionamiento la comprobación de paridad de `interflib.dimensional.parity` y su uso en
`tools/run_calibration.py`. El problema que resuelve tiene dos causas, y ninguna se ve en la imagen:

1. **El sentido del orden de interferencia** (`order_sign`, $s_\kappa$). El coseno es par, así que una sola toma no
   distingue hacia dónde crece el orden. Leída al revés, la fracción sale como $1-F$.
2. **El lado de la platina.** Los lectores comparan la fase del bloque con la de la platina en la columna del borde.
   Si se declara la platina en el lado equivocado, la diferencia cambia de signo y se lee otra vez $1-F$.

Las dos tienen el mismo efecto, así que solo cuenta su producto, la **paridad** de la lectura, y dos errores se
cancelan. El cuaderno recorre:

1. el problema, con escenas simuladas de verdad conocida;
2. los dos peines de candidatos de longitud y la distancia mínima entre ellos;
3. `check_fraction_parity` sin valor previo y con él, con sus cuatro veredictos;
4. cuándo puede decidir la prueba, según $F$ y la incertidumbre del previo;
5. la medición de extremo a extremo con `run_calibration`;
6. opcionalmente, un bloque real certificado.

**Convenciones.** Longitudes y longitudes de onda en nanómetros. $L = (\kappa + F)\,\lambda/2$ es la longitud del
bloque en semiondas [Doiron & Beers, *The Gage Block Handbook*, NIST Monograph 180 (1995), §6.5]; $d = L - L'$ es la
desviación respecto al nominal $L'$.

## 0. Preparación

In [ ]:
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from interflib.dimensional.parity import check_fraction_parity
from interflib.fringe.fourier import fourier_edge_fraction
from interflib.simulation import PhaseModel, SceneGeometry, SensorModel, synthesize_interferogram

plt.rcParams.update({"figure.dpi": 110, "image.cmap": "gray", "font.size": 10})

# run_calibration is a script in tools/, not part of the package: load it as a module
TOOL = Path.cwd().parent / "tools" / "run_calibration.py"
spec = importlib.util.spec_from_file_location("run_calibration", TOOL)
run_calibration = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run_calibration)

WAVELENGTH = 543.5          # wavelength in air of the simulated scenes, nm
HALF = WAVELENGTH / 2
NOMINAL = 2e6               # 2 mm block, nm
TRUE_DEV = 51.0             # true deviation from nominal, nm

## 1. El problema: una toma no distingue $F$ de $1-F$

Se simulan dos escenas del mismo bloque, de 2 mm + 51 nm. Solo cambia el sentido de la inclinación del espejo de
referencia: 30 franjas a lo largo de $y$ en un caso y −30 en el otro. El orden crece hacia $+y$ en la primera y hacia
$-y$ en la segunda, pero las dos imágenes son casi la misma: el coseno es par.

In [ ]:
geometry = SceneGeometry.cs505mu(scale=0.5, fov_mm=25.0)


def scene(fringes_y, seed=0):
    model = PhaseModel(wavelength_nm=WAVELENGTH, central_length_nm=NOMINAL + TRUE_DEV,
                       fringes_y=fringes_y, fringes_x=1.0)
    return synthesize_interferogram(geometry, model, sensor=SensorModel.cs505mu(), seed=seed)


up, down = scene(+30.0), scene(-30.0)
mask = up.block_mask
height, width = mask.shape
eval_row = height // 2
edge_x = float(np.flatnonzero(mask[eval_row])[0]) - 0.5      # first block column at the evaluation row
rows = (eval_row - 75, eval_row + 75)                       # fit window along y
fft_rows = (max(0, eval_row - 450), min(height, eval_row + 450))   # Fourier span, as in run_calibration
print(f"image {height} x {width} px, edge at x = {edge_x:.1f} px, true F = {up.fraction_truth:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, sim, title in zip(axes, (up, down), ("orden creciente hacia +y", "orden creciente hacia −y")):
    ax.imshow(sim.image, extent=(0, width, height, 0))
    ax.axvline(edge_x, color="tab:orange", lw=1)
    ax.set_xlim(edge_x - 200, edge_x + 200)
    ax.set_ylim(eval_row + 150, eval_row - 150)
    ax.set_title(title)
    ax.set_xlabel("columna x (px)")
axes[0].set_ylabel("fila y (px)")
fig.tight_layout()

Los lectores devuelven la verdad solo si se les declara el sentido correcto. Con el equivocado devuelven $1-F$:

In [ ]:
def read(img, x_edge, order_sign=1):
    return fourier_edge_fraction(img, x_edge, rows, eval_row, fft_rows=fft_rows, order_sign=order_sign).fraction


img_up, img_down = up.image.astype(float), down.image.astype(float)
print(f"true F                          {up.fraction_truth:.4f}")
print(f"+y scene, order_sign +1         {read(img_up, edge_x, +1):.4f}")
print(f"-y scene, order_sign +1 (wrong) {read(img_down, edge_x, +1):.4f}   1 - F = {1 - up.fraction_truth:.4f}")
print(f"-y scene, order_sign -1         {read(img_down, edge_x, -1):.4f}")

**El lado de la platina.** Los lectores suponen la platina a la izquierda del borde. Si la imagen se invierte en $x$,
la platina queda a la derecha. Leerla como si siguiera a la izquierda (el error de declarar mal el lado) da otra vez
$1-F$. Y si además se declara mal el sentido, los dos errores se cancelan:

In [ ]:
mirrored = img_up[:, ::-1]
edge_mirrored = width - 1 - edge_x     # the same edge in the mirrored image (platen now on its right)
print(f"platen side wrong, order_sign +1          {read(mirrored, edge_mirrored, +1):.4f}   (1 - F)")
print(f"platen side wrong, order_sign -1 (wrong)  {read(mirrored, edge_mirrored, -1):.4f}   (F: the errors cancel)")

Lo que importa, entonces, es el producto $s_\kappa \cdot s_\text{lado}$: la paridad de la lectura. La imagen no la
fija, porque la fase de bloque y platina es la misma con cualquiera de las dos declaraciones salvo el signo. Hay que
fijarla con información externa: un desplazamiento conocido o una longitud conocida de antemano.

## 2. Dos peines de candidatos

Con una sola longitud de onda, cada lectura de la fracción admite una familia de longitudes separadas $\lambda/2$,
una por cada orden entero $\kappa$:

$$d_m(F) = d_0(F) + m\,\frac{\lambda}{2}, \qquad m \in \mathbb{Z}.$$

Como $L = (\kappa+F)\lambda/2$, el peine de $1-F$ es el de $F$ desplazado:

$$d_m(1-F) = d_m(F) + (1 - 2F)\,\frac{\lambda}{2}.$$

El orden se resuelve tomando en cada peine el candidato más cercano a un valor previo: el nominal ($d = 0$) o una
medición anterior [Rojas-Rodríguez, Galvis Arroyave & Vinck-Posada, preprint INM (2026), §III.C]. La distancia
mínima entre los dos peines es

$$\Delta_\text{min} = \frac{\lambda}{2}\,\bigl|2F - \operatorname{round}(2F)\bigr|,$$

y es lo mínimo que una paridad equivocada movería el resultado.

In [ ]:
F = up.fraction_truth
check = check_fraction_parity(HALF, TRUE_DEV, F)
m = np.arange(-2, 3)
comb_declared = check.length_dev + m * HALF
comb_flipped = check.flipped_length_dev + m * HALF

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1.6, 1]})
ax.vlines(comb_declared, 0, 1, color="tab:blue", lw=2, label=f"lectura $F$ = {F:.3f}")
ax.vlines(comb_flipped, 0, 1, color="tab:red", lw=2, ls="--", label=f"lectura $1-F$ = {1 - F:.3f}")
ax.axvline(0, color="k", lw=0.8, ls=":", label="nominal")
ax.annotate("", xy=(check.flipped_length_dev, 0.5), xytext=(check.length_dev, 0.5),
            arrowprops=dict(arrowstyle="<->", color="k"))
ax.text((check.length_dev + check.flipped_length_dev) / 2, 0.62, f"{check.shift:+.1f} nm", ha="center",
        bbox=dict(fc="white", ec="none", pad=1))
ax.set_xlim(-2.2 * HALF, 2.2 * HALF)
ax.set_yticks([])
ax.set_xlabel("desviación respecto al nominal d (nm)")
ax.set_title("candidatos de las dos lecturas")
ax.legend(fontsize=8, loc="upper right")

fs = np.linspace(0, 1, 401)
ax2.plot(fs, [check_fraction_parity(HALF, 0.0, f).separation for f in fs], color="k")
ax2.axhline(HALF / 2, color="0.6", lw=0.8, ls=":")
ax2.set_xlabel("fracción F")
ax2.set_ylabel(r"$\Delta_\mathrm{min}$ (nm)")
ax2.set_title("distancia mínima entre los peines")
fig.tight_layout()
print(f"declared d = {check.length_dev:+.1f} nm, flipped d = {check.flipped_length_dev:+.1f} nm, "
      f"minimum separation {check.separation:.1f} nm, verdict: {check.verdict}")

La distancia mínima es una onda triangular: llega a $\lambda/4$ en $F = 0{,}25$ y $0{,}75$, y se anula en $F = 0$ y
$0{,}5$. Con esas fracciones, $F$ y $1-F$ son la misma lectura (o difieren en una franja entera) y equivocarse de
paridad no cambia nada. Sin valor previo la paridad queda `unverified`, pero la desviación de la otra lectura dice qué
está en juego.

## 3. La prueba contra un valor previo

Con un valor previo $d_p$ y su incertidumbre típica $u_p$ (el certificado de un patrón de control o la premedición
mecánica), cada candidato tiene una distancia normalizada

$$z = \frac{d - d_p}{\sqrt{u_p^2 + u_\text{opt}^2}},$$

donde $u_\text{opt}$ es la incertidumbre de la desviación óptica misma. Un candidato es compatible si $|z| \le 3$
[sin fuente — criterio propio]. Hay cuatro veredictos:

| | otra lectura compatible | otra lectura no compatible |
|---|---|---|
| **declarada compatible** | `undecided` | `confirmed` |
| **declarada no compatible** | `contradicted` | `inconsistent` |

Los cuatro casos, con la lectura de la escena simulada:

In [ ]:
cases = {
    "confirmed":    dict(prior_length_dev=51.0, prior_u=10.0),
    "contradicted": dict(prior_length_dev=check.flipped_length_dev, prior_u=10.0),
    "undecided":    dict(prior_length_dev=0.0, prior_u=60.0),
    "inconsistent": dict(prior_length_dev=check.length_dev + HALF / 2, prior_u=5.0),
}
fig, axes = plt.subplots(len(cases), 1, figsize=(8, 6.4), sharex=True)
d = np.linspace(-2.2 * HALF, 2.2 * HALF, 1200)
for ax, (name, prior) in zip(axes, cases.items()):
    c = check_fraction_parity(HALF, TRUE_DEV, F, **prior)
    u = prior["prior_u"]
    ax.fill_between(d, np.exp(-0.5 * ((d - prior["prior_length_dev"]) / u) ** 2), color="0.85")
    ax.vlines(comb_declared, 0, 1, color="tab:blue", lw=2)
    ax.vlines(comb_flipped, 0, 1, color="tab:red", lw=2, ls="--")
    ax.set_yticks([])
    ax.set_title(f"{c.verdict}: previo {prior['prior_length_dev']:+.0f} ± {u:.0f} nm;  "
                 f"z = {c.z:+.2f} (F), {c.flipped_z:+.2f} (1 − F);  log10 LR = {c.log10_likelihood_ratio:+.1f}",
                 fontsize=9)
axes[-1].set_xlabel("desviación respecto al nominal d (nm)")
fig.tight_layout()

`log10 LR` es el logaritmo de la razón de verosimilitud gaussiana de la lectura declarada frente a la otra. Es
positivo cuando el previo favorece la lectura declarada.

- `confirmed`: el previo coincide con la lectura declarada y excluye la otra.
- `contradicted`: el previo coincide con la otra lectura. Es el aviso «WRONG PARITY» de `run_calibration`:
  `order_sign` o el lado de la platina están mal.
- `undecided`: el previo es demasiado ancho para separar los peines. El error posible es entonces del orden de la
  incertidumbre del previo.
- `inconsistent`: el previo cae entre los dos peines y lejos de ambos. No es un problema de paridad: hay que revisar
  $\lambda_0$, la presión, el previo o el punto donde se evalúa $F$.

## 4. ¿Cuándo puede decidir la prueba?

Si la paridad declarada es la correcta y el previo está centrado en la verdad, la otra lectura queda a
$\Delta_\text{min}/\sigma$ desviaciones típicas, con $\sigma = \sqrt{u_p^2 + u_\text{opt}^2}$. La prueba decide
cuando eso supera 3. El mapa muestra la zona en que decide, según la fracción leída y la incertidumbre del previo
($u_\text{opt} = 0$):

In [ ]:
fs = np.linspace(0, 1, 201)
us = np.linspace(2, 60, 117)
verdicts = np.array([[check_fraction_parity(HALF, TRUE_DEV, f, prior_length_dev=TRUE_DEV, prior_u=u).verdict
                      == "confirmed" for f in fs] for u in us])
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.pcolormesh(fs, us, verdicts, cmap=ListedColormap(["white", "tab:green"]), vmin=0, vmax=1, shading="auto")
ax.plot(fs, [check_fraction_parity(HALF, 0.0, f).separation / 3 for f in fs], color="k", lw=1)
ax.set_xlabel("fracción leída F")
ax.set_ylabel("u del previo (nm)")
ax.set_title(r"verde: confirmed;  blanco: undecided  (frontera $u = \Delta_\mathrm{min}/3$)")
fig.tight_layout()

Con la incertidumbre de un certificado ($U = 20$ nm, $u = 10$ nm), la prueba decide salvo a menos de unas 0,06 de
$F = 0$ y de $F = 0{,}5$, donde de todos modos el error posible es pequeño (menos de 30 nm). Con un previo de $u \approx 45$
nm ya no decide nunca, porque $\Delta_\text{min} \le \lambda/4 \approx 136$ nm. **Sin un previo con incertidumbre
la prueba no es posible**: la paridad queda como control del procedimiento, fijada con un patrón certificado después
de cada realineación o con un desplazamiento conocido.

## 5. De extremo a extremo con `run_calibration`

El script corre la cadena completa sobre una escena simulada (por defecto, el bloque de 2 mm + 51 nm) y añade la
sección «Parity» al resumen. Tres corridas:

1. sentido correcto, sin previo: `unverified`, con la longitud de la otra lectura en el aviso;
2. sentido correcto, con el previo del certificado: `confirmed`;
3. referencia inclinada al revés (`--fringes-y -30`) y `order_sign` +1 declarado: con el previo, `contradicted`.

El script usa la longitud de onda en aire de la ecuación de Edlén (543,406 nm con sus condiciones por defecto), no los
543,5 nm de las secciones anteriores. Por eso $F$ cambia, aunque la desviación simulada sigue siendo +51 nm.

In [ ]:
SMALL = ["--scale", "0.25", "--frames", "2"]
PRIOR = ["--prior-dev-nm", "51", "--prior-U-nm", "20"]


def show(argv):
    summary = run_calibration.run(run_calibration.parse_args([*SMALL, *argv]))
    text = run_calibration.format_summary(summary).splitlines()
    start = next(i for i, line in enumerate(text) if line.startswith("Parity"))
    print("\n".join([text[3], *text[start - 1:start + 4]]))
    for warning in summary["warnings"]:
        if "parity" in warning.lower() or "PARITY" in warning:
            print("  warning:", warning)
    print("-" * 78)
    return summary


plain = show([])
confirmed = show(PRIOR)
wrong = show(["--fringes-y", "-30", "--order-sign", "1", *PRIOR])

En la tercera corrida la longitud principal sale equivocada, unos 100 nm por debajo de la verdad. Con el previo,
la sección de paridad lo detecta y la otra lectura recupera la verdad: la desviación de 51 nm de la escena.

In [ ]:
print(f"true deviation                {TRUE_DEV:+.1f} nm")
print(f"declared reading (wrong sign) {wrong['length']['deviation_from_nominal_nm']:+.1f} nm")
print(f"flipped reading               {wrong['parity']['flipped_length_dev_nm']:+.1f} nm  "
      f"-> {wrong['parity']['verdict']}")

## 6. Opcional: un bloque real certificado

Si está disponible la carpeta de datos del proyecto, se mide el bloque de 2 mm de la campaña de 2026-10 (serie
`repet-2`, 30 cuadros). Su certificado de calibración da $d = +51 \pm 20$ nm ($k = 2$) [CENAM, certificado
CNM-CC-740-606/2023]. $F$ se lleva al centro de la cara (columna 1340,5), donde el certificado define la longitud.

**Supuestos**, todos marcados por el script:

- $\lambda_0 = 543{,}5163$ nm, sin fuente verificada;
- presión de 75,3 kPa, supuesta porque la campaña no la midió;
- $u_\text{opt} = 10$ nm, una hipótesis de trabajo para el resto de la incertidumbre óptica.

No es una calibración, sino una comprobación de que la prueba se comporta en datos reales.

In [ ]:
SERIES = (Path.cwd().parents[2] / "data" / "raw" / "202610_estudio-fraccion" / "repetibilidad"
          / "20261005_L1_tg-inm_2mm_repet-2_serie-010")
if SERIES.exists():
    common = ["--series", str(SERIES), "--pressure", "75300", "--wavelength-vac", "543.5163",
              "--face-centre-x", "1340.5", "--u-other-nm", "10", *PRIOR]
    for sign in ("1", "-1"):
        s = run_calibration.run(run_calibration.parse_args([*common, "--order-sign", sign]))
        p = s["parity"]
        print(f"order_sign {int(sign):+d}: d = {p['length_dev_nm']:+.1f} nm, other reading {p['flipped_length_dev_nm']:+.1f} nm; "
              f"z = {p['z']:+.2f} / {p['flipped_z']:+.2f} -> {p['verdict']}")
else:
    print(f"data folder not found ({SERIES}); this section is skipped")

## 7. Resumen

- El lado de la platina y el sentido del orden son **una sola paridad**. Cada error lee $1-F$, y dos errores se
  cancelan.
- Una toma no la determina. `check_fraction_parity` dice siempre qué daría la otra lectura: al menos
  $\Delta_\text{min} = (\lambda/2)\,|2F - \operatorname{round}(2F)|$ de diferencia.
- Con un valor previo y su incertidumbre, la prueba confirma o contradice la paridad declarada cuando
  $\Delta_\text{min}$ supera unas tres veces la incertidumbre combinada.
- Sin previo, la paridad no se puede probar en la medición. Hay que fijarla en el banco con un patrón certificado
  después de cada realineación, o con un desplazamiento conocido.
- La prueba solo vale si $F$ se evalúa donde el previo define la longitud: el centro de la cara para un certificado.